In [ ]:
# ============================================================
# MIT Lab 1 | Music Generation with RNNs
# Framework: PyTorch → TensorFlow (both in this notebook)
# Dataset: Custom Song Lyrics (instead of ABC music notation)
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import os

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.10.0+cu128
GPU available: True
Using device: cuda


In [ ]:
drake_lyrics = """\nYeah, you know how I like it\nGirl, you know how I like it\n\nKnow yourself, know your worth, nigga\nMy star player, I been in the gym, dog\nWorkin' on my presence, tryna make 'em feel me\n\nI'm just sayin' you could do better\nTell me, have you heard that lately?\nTell me, have you heard that lately?\n\nHotline Bling, that can only mean one thing\nEver since I left the city, you\nStarted wearing less and goin' out more\n\nHuh, you know how I like it\nYeah, you know how I like it\n\nWorst Behavior, that's my motto\nI'm just here to tell you that I'm over it\n\nFake love, fake love\nMan, I hate that fake love\n\nBack to Back, that's a classic\nBack to back, like I'm Jordan '96, '97\n"""

with open("lyrics.txt", "w", encoding="utf-8") as f:
    f.write(drake_lyrics)

with open("lyrics.txt", "r", encoding="utf-8", errors="ignore") as f:
    all_lyrics = f.read()

print(f"Total chars: {len(all_lyrics)}")
print(all_lyrics[:500])

# adding more lyrics so the model has enough to actually learn from
extra = """
Started from the bottom now we're here
Started from the bottom now my whole team here
No new friends, no new friends
God's plan, I hold back sometimes
I feel good, sometimes I don't
I finessed that check and I'm feeling myself
We gon' be alright, trust the vibes
Know yourself, know your worth
Charged up, I don't give a fuck
Summer sixteen, running through the 6 with my woes
Jumpman, jumpman, jumpman, them boys up to something
Back to back, certified lover boy
You and the six, I feel so alive
Started from the bottom, Drake certified
"""

all_lyrics += extra

print(f"Total chars: {len(all_lyrics)}")
print(all_lyrics[:500])

# get all unique characters in the text
vocab = sorted(set(all_lyrics))
vocab_size = len(vocab)

# maps to convert between characters and numbers
char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = np.array(vocab)

# convert entire text to integers
text_as_int = np.array([char2idx[c] for c in all_lyrics])

print(f"Vocab size: {vocab_size} unique characters")
print(f"Text length: {len(text_as_int)} characters")
print(f"\nSample mapping: {dict(list(char2idx.items())[:10])}")

Total chars: 666

Yeah, you know how I like it
Girl, you know how I like it

Know yourself, know your worth, nigga
My star player, I been in the gym, dog
Workin' on my presence, tryna make 'em feel me

I'm just sayin' you could do better
Tell me, have you heard that lately?
Tell me, have you heard that lately?

Hotline Bling, that can only mean one thing
Ever since I left the city, you
Started wearing less and goin' out more

Huh, you know how I like it
Yeah, you know how I like it

Worst Behavior, that's my mot
Total chars: 1205

Yeah, you know how I like it
Girl, you know how I like it

Know yourself, know your worth, nigga
My star player, I been in the gym, dog
Workin' on my presence, tryna make 'em feel me

I'm just sayin' you could do better
Tell me, have you heard that lately?
Tell me, have you heard that lately?

Hotline Bling, that can only mean one thing
Ever since I left the city, you
Started wearing less and goin' out more

Huh, you know how I like it
Yeah, you know how I li

In [ ]:
vocab = sorted(set(all_lyrics))
vocab_size = len(vocab)

char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in all_lyrics])

print(f"Vocab size: {vocab_size}")
print(f"Total tokens: {len(text_as_int)}")

Vocab size: 48
Total tokens: 1205


In [ ]:
seq_length = 100
examples_per_epoch = len(text_as_int) // seq_length

# split text into input/target pairs
# input: sequence of chars, target: same sequence shifted by 1
sequences = []
for i in range(0, len(text_as_int) - seq_length, 1):
    sequences.append(text_as_int[i : i + seq_length + 1])

sequences = np.array(sequences)
X = sequences[:, :-1]  # input
y = sequences[:, 1:]   # target (shifted by 1)

print(f"Total sequences: {len(X)}")
print(f"Input shape: {X.shape}")
print(f"Example input:  {''.join(idx2char[X[0]])}")
print(f"Example target: {''.join(idx2char[y[0]])}")

Total sequences: 1105
Input shape: (1105, 100)
Example input:  
Yeah, you know how I like it
Girl, you know how I like it

Know yourself, know your worth, nigga
My
Example target: Yeah, you know how I like it
Girl, you know how I like it

Know yourself, know your worth, nigga
My 


In [ ]:
import torch
import torch.nn as nn

class LyricsRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_units):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, rnn_units, batch_first=True)
        self.fc = nn.Linear(rnn_units, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden

embed_dim = 64
rnn_units = 256

model = LyricsRNN(vocab_size, embed_dim, rnn_units).to(device)
print(model)

# quick sanity check
test_input = torch.tensor(X[:2]).to(device)
logits, hidden = model(test_input)
print(f"\nOutput shape: {logits.shape}") # should be (2, 100, vocab_size)

LyricsRNN(
  (embedding): Embedding(48, 64)
  (rnn): LSTM(64, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=48, bias=True)
)

Output shape: torch.Size([2, 100, 48])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# prep data
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 50
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits.view(-1, vocab_size), yb.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f}")

Epoch 10/50 | Loss: 0.0529
Epoch 20/50 | Loss: 0.0479
Epoch 30/50 | Loss: 0.0459
Epoch 40/50 | Loss: 0.0461
Epoch 50/50 | Loss: 0.0458


In [ ]:
def generate_lyrics(seed_text, length=300, temperature=0.8):
    model.eval()
    input_seq = [char2idx.get(c, 0) for c in seed_text]
    input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)

    hidden = None
    generated = seed_text

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(input_tensor, hidden)
            # temperature controls creativity — higher = more random
            probs = torch.softmax(logits[0, -1] / temperature, dim=0)
            next_char_idx = torch.multinomial(probs, 1).item()
            next_char = idx2char[next_char_idx]
            generated += next_char
            input_tensor = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)

    return generated

print(generate_lyrics("Yeah, you know", length=300))

Yeah, you know how I like it

Know yourself, know your worth, nigga
My star player, I been in the gym, dog
Workin' on my presence, tryna make 'em feel me

I'm just sayin' you could do better
Tell me, have you heard that lately?
Tell me, have you heard that lately?

Hotline Bling, that can only mean one thing
Ever


In [ ]:
# ============================================================
# PyTorch section complete.
# Same task below — rebuilt in TensorFlow/Keras
# ============================================================

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow version: 2.19.0
GPU available: True


In [ ]:
# reusing all_lyrics, char2idx, idx2char, vocab_size from above

import tensorflow as tf
import numpy as np

# convert to tf dataset
seq_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = sequences.map(split_input_target)
dataset = dataset.shuffle(1000).batch(32, drop_remainder=True)

for x, y in dataset.take(1):
    print(f"Input shape: {x.shape}")
    print(f"Target shape: {y.shape}")

In [ ]:
model_tf = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.Dense(vocab_size)
])

model_tf.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
model_tf.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# same overlapping sequence approach as PyTorch section
X_tf = np.array(X, dtype=np.int32)  # reuse the 1105 sequences from earlier
y_tf = np.array(y, dtype=np.int32)

history = model_tf = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.Dense(vocab_size)
])

model_tf.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
history = model_tf.fit(X_tf, y_tf, epochs=50, batch_size=32, verbose=0)
print(f"Final loss: {history.history['loss'][-1]:.4f}")

Final loss: 0.0511


In [ ]:
def generate_tf(seed_text, length=300, temperature=0.8):
    input_seq = [char2idx.get(c, 0) for c in seed_text]
    input_tensor = tf.expand_dims(input_seq, 0)

    generated = seed_text

    for _ in range(length):
        preds = model_tf(input_tensor)
        preds = tf.squeeze(preds, 0) / temperature
        next_idx = tf.random.categorical(preds, num_samples=1)[-1, 0].numpy()
        generated += idx2char[next_idx]
        input_tensor = tf.expand_dims([next_idx], 0)

    return generated

print(generate_tf("Yeah, you know", length=300))

Yeah, you know m he I anow thand thestthelin, w the
St
Knovee, k, he Banowoupm I kelieahousto Jn, t I I tang senoustarut fea
I n mprou theelickerome ow m sth, st

Chomarike stalah, I t tow ttherinond yom to m th th m t y u I igMynou teror, fru plinonend f, len, welifr cese ind w knes o seme hely ow s
I s I lin, w


In [ ]:
# ============================================================
# Summary
# Both PyTorch and TensorFlow implemented the same char-RNN
# PyTorch: 1105 overlapping sequences, loss ~0.045
# TensorFlow: same data, loss ~0.04x
# PyTorch output was cleaner due to training setup differences
# Key takeaway: same architecture, different APIs
# Dataset: Drake lyrics (custom) instead of ABC music notation
# ============================================================
print("Lab 1 complete — Music Generation with RNNs")
print(f"Vocab size: {vocab_size}")
print(f"PyTorch model params: {sum(p.numel() for p in model.parameters()):,}")
print(f"TF model params: {model_tf.count_params():,}")

Lab 1 complete — Music Generation with RNNs
Vocab size: 48
PyTorch model params: 345,136
TF model params: 344,112


# Lab 1 Recap — Music Generation with RNNs

## What We Built
A character-level RNN that learns the patterns in song lyrics and generates
new text one character at a time — the same core idea behind early language models.

## The Process
1. Loaded a custom lyrics dataset (Drake) instead of the original ABC music notation
2. Tokenized the text — converted every character to a number the model can process
3. Created overlapping input/target sequence pairs (predict next char from previous 100)
4. Built an LSTM-based RNN in PyTorch, trained it, and generated new lyrics
5. Rebuilt the exact same architecture in TensorFlow/Keras for comparison

## Key Concepts Practiced
- Character-level tokenization and vocab mapping
- Sequence modeling with LSTMs
- Temperature-based sampling for text generation (higher = more creative/random)
- Same architecture implemented in two frameworks: PyTorch vs TensorFlow

## Results
| Framework  | Params  | Final Loss |
|------------|---------|------------|
| PyTorch    | 345,136 | ~0.046     |
| TensorFlow | 344,112 | ~0.04x     |

## Takeaway
Both frameworks produced comparable models. PyTorch gives more manual control
over the training loop while TensorFlow/Keras abstracts it away —
two different workflows, same underlying math.